In [63]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction import DictVectorizer

from sklearn.tree import DecisionTreeClassifier

In [2]:
df_full_data = pd.read_csv("data/credit_score/CreditScoring.csv")

In [13]:
df_full_data.head()

,status,seniority,home,time,age,marital,records,job,expenses,income,assets,debt,amount,price
0,ok,9,rent,60,30,married,no,freelance,73,129,0,0,800,846
1,ok,17,rent,60,58,widow,no,fixed,48,131,0,0,1000,1658
2,default,10,owner,36,46,married,yes,freelance,90,200,3000,0,2000,2985
3,ok,0,rent,60,24,single,no,fixed,63,182,2500,0,900,1325
4,ok,0,rent,36,26,single,no,fixed,46,107,0,0,310,910


In [4]:
df_full_data.columns = df_full_data.columns.str.lower()

In [9]:
df_full_data.home.value_counts()

home
2    2107
1     973
5     783
6     319
3     247
4      20
0       6
Name: count, dtype: int64

In [7]:
status_values = {
    1: "ok", 
    2: "default",
    0: "unk"
}

df_full_data.status = df_full_data.status.map(status_values)

In [11]:
home_values = {
    1: "rent", 
    2: "owner", 
    3: "private", 
    4: "ignore", 
    5: "parents", 
    6: "other", 
    0: "unk"
}

marital_values = {
    1: "single", 
    2: "married", 
    3: "widow", 
    4: "separated",
    5: "divorced",
    0: "unk"
}

record_values = {
    1: "no",
    2: "yes",
    0: "unk"
}

job_values = {
    1: "fixed",
    2: "parttime",
    3: "freelance",
    4: "others",
    0: "unk",
}

In [12]:
df_full_data.home = df_full_data.home.map(home_values)
df_full_data.marital = df_full_data.marital.map(marital_values)
df_full_data.records = df_full_data.records.map(record_values)
df_full_data.job = df_full_data.job.map(job_values)

In [18]:
numerical_col = df_full_data.describe().columns

In [25]:
error_num_col = numerical_col[(df_full_data[numerical_col].max() >= 99999999)]

In [26]:
for c in error_num_col:
    df_full_data[c] = df_full_data[c].replace(99999999, value=np.nan)

In [27]:
df_full_data[numerical_col].max() >= 99999999

seniority    False
time         False
age          False
expenses     False
income       False
assets       False
debt         False
amount       False
price        False
dtype: bool

In [30]:
df_full_data[error_num_col].max()

income       959.0
assets    300000.0
debt       30000.0
dtype: float64

In [31]:
df_full_data = df_full_data[df_full_data.status != "unk"].reset_index(drop=True)

In [32]:
df_full_data.status.unique()

<StringArray>
['ok', 'default']
Length: 2, dtype: str

In [34]:
df_y = df_full_data.status.values

In [57]:
df_train, df_test, y_train, y_test = train_test_split(df_full_data, df_y, test_size=0.4, random_state=1)

In [58]:
df_train.shape, df_test.shape, y_train.shape, y_test.shape

((2672, 14), (1782, 14), (2672,), (1782,))

In [59]:
df_test, df_val, y_test, y_val = train_test_split(df_test, y_test, test_size=0.5, random_state=1)

In [60]:
df_test.shape, df_val.shape, y_test.shape, y_val.shape

((891, 14), (891, 14), (891,), (891,))

In [61]:
del df_train["status"]
del df_test["status"]
del df_val["status"]

In [62]:
df_train

,seniority,home,time,age,marital,records,job,expenses,income,assets,debt,amount,price
1906,20,owner,12,43,married,no,freelance,90,0.0,3000.0,0.0,500,1909
1924,10,owner,60,40,married,no,fixed,105,125.0,6500.0,0.0,1500,1800
2038,20,rent,24,40,married,no,fixed,87,117.0,0.0,0.0,350,350
1737,5,rent,36,39,married,yes,fixed,100,250.0,0.0,0.0,1400,3959
2476,13,parents,48,28,married,no,fixed,90,105.0,0.0,0.0,1200,1590
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2895,31,owner,48,57,married,yes,freelance,60,230.0,19400.0,1781.0,600,887
2763,19,owner,60,46,married,no,freelance,60,190.0,7000.0,0.0,1500,2755
905,15,owner,60,55,married,no,freelance,60,0.0,6500.0,3650.0,1200,1710
3980,12,rent,48,36,married,no,fixed,80,195.0,0.0,0.0,1800,2272


In [115]:
def data_transform(vectors, dv=None):
    
    dict_vectors = vectors.fillna(0).to_dict(orient="records")

    if dv is None:
        dv = DictVectorizer()
        v_transform = dv.fit_transform(dict_vectors)
        return dv, v_transform
    else:
        v_transform = dv.transform(dict_vectors)
        return v_transform

In [116]:
dt = DecisionTreeClassifier()

In [117]:
dv, x_train = data_transform(df_train)
dt.fit(x_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [118]:
train_pred = dt.predict_proba(x_train)[:, 1]

In [119]:
roc_auc_score(y_train, train_pred)

1.0

In [120]:
x_val = data_transform(df_val, dv)

In [121]:
dv.get_feature_names_out()

array(['age', 'amount', 'assets', 'debt', 'expenses', 'home=ignore',
       'home=other', 'home=owner', 'home=parents', 'home=private',
       'home=rent', 'home=unk', 'income', 'job=fixed', 'job=freelance',
       'job=others', 'job=parttime', 'job=unk', 'marital=divorced',
       'marital=married', 'marital=separated', 'marital=single',
       'marital=unk', 'marital=widow', 'price', 'records=no',
       'records=yes', 'seniority', 'time'], dtype=object)

In [122]:
val_pred = dt.predict_proba(x_val)[:, 1]

roc_auc_score(y_val, val_pred)

0.6498909636091046

In [99]:
x_train.shape, x_val.shape

((2672, 29), (891, 28))